Weighting Feature Main Data


In [50]:
import pandas as pd
import ast
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
# Load the preprocessed data
file_path = '../data/preprocessed_data.csv'

df = pd.read_csv(file_path)
df.head()


,id_data,sentiment_label,review,processed_text,processed_tokens
0,1,positive,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...,"['one', 'reviewer', 'mentioned', 'watching', '..."
1,2,positive,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,"['wonderful', 'little', 'production', 'filming..."
2,3,positive,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,"['thought', 'wonderful', 'way', 'spend', 'time..."
3,4,negative,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,"['basically', 'family', 'little', 'boy', 'jake..."
4,5,positive,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,"['petter', 'matteis', 'love', 'time', 'money',..."


In [27]:
# Vectorize (TF-IDF)
vectorizer = TfidfVectorizer(min_df=5, max_features=10000, ngram_range=(1, 2))
x_features = vectorizer.fit_transform(df['processed_text'])

print("Extraction Feature (Count Row, Count Column):", x_features.shape)

Extraction Feature (Count Row, Count Column): (50000, 10000)


In [28]:
# Create Data Identity DataFrame (id_data, sentiment)
# Use reset_index to make sure the index is continous
df_identity = df[['id_data', 'sentiment_label']].reset_index(drop=True)

df_identity.head()

,id_data,sentiment_label
0,1,positive
1,2,positive
2,3,positive
3,4,negative
4,5,positive


In [29]:
# Create Sparse DataFrame (Dimension Table) from the TF-IDF matrix
df_features = pd.DataFrame.sparse.from_spmatrix(
    x_features, 
    columns=vectorizer.get_feature_names_out()
)

df_features.head()

,aaron,abandon,abandoned,abbott,abc,ability,able,able get,able make,able see,...,youthful,youtube,yr,zane,zany,zero,zombie,zombie movie,zone,zoom
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.105413,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Concatenate Table Identity and Feature horizontally (axis=1)
df_generated = pd.concat([df_identity, df_features], axis=1)

print("Dimension Table:", df_generated.shape)
df_generated.head()

Dimension Table: (50000, 10002)


,id_data,sentiment_label,aaron,abandon,abandoned,abbott,abc,ability,able,able get,...,youthful,youtube,yr,zane,zany,zero,zombie,zombie movie,zone,zoom
0,1,positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,negative,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.105413,NaN,NaN,NaN
4,5,positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# Validation Weights & Match Check
selected_id = 4

selected_data_text = df['id_data'] == selected_id

text_data = df.loc[selected_data_text, 'processed_text'].values[0]
raw_tokens_data = df.loc[selected_data_text, 'processed_tokens'].values[0]

# Check if the raw_tokens_data is a string representation of a list, and convert it
if isinstance(raw_tokens_data, str):
    tokens_data = ast.literal_eval(raw_tokens_data)
else:
    tokens_data = raw_tokens_data

selected_data_dim = df_generated['id_data'] == selected_id
dim_data = df_generated[selected_data_dim]

# Drop the 'id_data' and 'sentiment_label' columns to get only the weights
weights_only_columns = dim_data.drop(columns=['id_data', 'sentiment_label']).iloc[0]

# Drop NaN/0 values and sort the weights in descending order
valid_weights = weights_only_columns.dropna().sort_values(ascending=False)

# Match the tokens in the document with the valid weights
tokens_in_doc = set(tokens_data) if isinstance(tokens_data, list) else set() 

# Filtering matched features
matched_weights = valid_weights[valid_weights.index.isin(tokens_in_doc)]

print(f"Processed text for ID {selected_id}: {text_data}\n")
print(f"Total unique tokens in document text: {len(tokens_in_doc)}")
print(f"Total active TF-IDF features (Unigrams & Bigrams): {len(valid_weights)}\n")

if len(matched_weights) > 0:
    print(f"Validation Status: SUCCESS (Match Found!)")
    print(f"Total matched features: {len(matched_weights)}\n")
    print("Top Matched Features and their TF-IDF Weights:")
    print(matched_weights.head(10))
else:
    print("Validation Status: FAILED (No matched features found.)")

Processed text for ID 4: basically family little boy jake think zombie closet parent fighting timethis movie slower soap opera suddenly jake decides become rambo kill zombieok first going make film must decide thriller drama drama movie watchable parent divorcing arguing like real life jake closet totally ruin film expected see boogeyman similar movie instead watched drama meaningless thriller spot well playing parent descent dialog shot jake ignore

Total unique tokens in document text: 51
Total active TF-IDF features (Unigrams & Bigrams): 54

Validation Status: SUCCESS (Match Found!)
Total matched features: 46

Top Matched Features and their TF-IDF Weights:
jake           0.548683
parent         0.276721
closet         0.269161
drama          0.243448
thriller       0.183322
rambo          0.150927
arguing        0.145647
descent        0.136491
meaningless    0.129842
ignore         0.121303
Name: 3, dtype: Sparse[float64, nan]


In [51]:
# Export Weighted Dimension Table to CSV
output_file_path = '../data/weighted_dimension_table.csv'
chunk_size = 5000
total_rows = len(df_generated)

# Write header to CSV file first
df_generated.iloc[0:0].to_csv(output_file_path, index=False)

# Looping with tqdm bar
for start in tqdm(range(0, total_rows, chunk_size), desc="Exporting Progress"):
    end = min(start + chunk_size, total_rows)
    chunk = df_generated.iloc[start:end]
    
    # Append chunk to CSV file without header
    chunk.to_csv(output_file_path, mode='a', index=False, header=False)

print(f"\nSuccessfully exported to: {output_file_path}")

Exporting Progress: 100%|██████████| 10/10 [1:10:15<00:00, 421.56s/it]


Successfully exported to: ../data/weighted_dimension_table.csv
